In [1]:
import pandas as pd

df = pd.read_csv("C://Users//HP//Datasets//buenos-aires-real-estate-1.csv")  # adjust to your actual file name/path

print(df.columns.tolist())
print(df.head())

['operation', 'property_type', 'place_with_parent_names', 'lat-lon', 'price', 'currency', 'price_aprox_local_currency', 'price_aprox_usd', 'surface_total_in_m2', 'surface_covered_in_m2', 'price_usd_per_m2', 'price_per_m2', 'floor', 'rooms', 'expenses', 'properati_url']
  operation property_type                            place_with_parent_names  \
0      sell     apartment           |Argentina|Capital Federal|Villa Crespo|   
1      sell         house  |Argentina|Bs.As. G.B.A. Zona Oeste|La Matanza...   
2      sell         house  |Argentina|Bs.As. G.B.A. Zona Oeste|Morón|Cast...   
3      sell         house  |Argentina|Bs.As. G.B.A. Zona Oeste|Tres de Fe...   
4      sell     apartment              |Argentina|Capital Federal|Chacarita|   

                         lat-lon     price currency  \
0  -34.6047834183,-58.4586812499  180000.0      USD   
1                            NaN  250000.0      USD   
2         -34.6497002,-58.658073  410000.0      USD   
3        -34.5957086,-58.5669

In [2]:
print(df.shape)
print(df.isnull().sum())

(8606, 16)
operation                        0
property_type                    0
place_with_parent_names          0
lat-lon                       1670
price                         1016
currency                      1016
price_aprox_local_currency    1016
price_aprox_usd               1016
surface_total_in_m2           2660
surface_covered_in_m2         1338
price_usd_per_m2              3711
price_per_m2                  2086
floor                         7347
rooms                         3854
expenses                      7731
properati_url                    0
dtype: int64


In [4]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

# ============================================
# 1. LOAD DATA
# ============================================
df = pd.read_csv("C://Users//HP//Datasets//buenos-aires-real-estate-1.csv")  # adjust filename/path as needed

# ============================================
# 2. CLEAN & EXTRACT NEW COLUMNS
# ============================================

# ---- Extract neighborhood from the pipe-separated location string ----
# Example string: "|Argentina|Capital Federal|Villa Crespo|"
# Splitting on '|' gives: ['', 'Argentina', 'Capital Federal', 'Villa Crespo', '']
# The neighborhood is usually the second-to-last non-empty piece.
def extract_neighborhood(location_string):
    parts = [p for p in location_string.split('|') if p != '']
    if len(parts) >= 1:
        return parts[-1]  # last meaningful piece = most specific location
    return 'Unknown'

df['neighborhood'] = df['place_with_parent_names'].apply(extract_neighborhood)

# ---- Split lat-lon into two separate numeric columns ----
# Only do this for rows where lat-lon actually exists
df[['lat', 'lon']] = df['lat-lon'].str.split(',', expand=True)
df['lat'] = pd.to_numeric(df['lat'], errors='coerce')
df['lon'] = pd.to_numeric(df['lon'], errors='coerce')

# ============================================
# 3. INITIALIZE APP
# ============================================
app = Dash(__name__)

# ============================================
# 4. LAYOUT
# ============================================
app.layout = html.Div([

    html.H1("Buenos Aires Real Estate Market Explorer",
            style={'textAlign': 'center', 'fontFamily': 'Georgia, serif', 'color': '#2c3e50'}),

    # ---- Filters ----
    html.Div([
        html.Div([
            html.Label("Property Type"),
            dcc.Dropdown(
                id='type-filter',
                options=[{'label': 'All Types', 'value': 'All'}] +
                        [{'label': t, 'value': t} for t in sorted(df['property_type'].unique())],
                value='All',
                clearable=False
            )
        ], style={'width': '45%', 'display': 'inline-block'}),

        html.Div([
            html.Label("Operation"),
            dcc.Dropdown(
                id='operation-filter',
                options=[{'label': 'All', 'value': 'All'}] +
                        [{'label': o, 'value': o} for o in sorted(df['operation'].unique())],
                value='All',
                clearable=False
            )
        ], style={'width': '45%', 'display': 'inline-block', 'marginLeft': '4%'}),
    ], style={'padding': '20px'}),

    # ---- KPI Cards ----
    html.Div(id='kpi-cards', style={'display': 'flex', 'justifyContent': 'space-around', 'padding': '10px'}),

    # ---- Map (the centerpiece) ----
    dcc.Graph(id='price-map'),

    # ---- Distribution + property type comparison ----
    html.Div([
        dcc.Graph(id='price-histogram', style={'width': '49%', 'display': 'inline-block'}),
        dcc.Graph(id='type-price-bar', style={'width': '49%', 'display': 'inline-block'}),
    ]),

    # ---- Top 10 neighborhoods ----
    dcc.Graph(id='neighborhood-bar'),

])


# ============================================
# 5. CALLBACK
# ============================================
@app.callback(
    Output('kpi-cards', 'children'),
    Output('price-map', 'figure'),
    Output('price-histogram', 'figure'),
    Output('type-price-bar', 'figure'),
    Output('neighborhood-bar', 'figure'),
    Input('type-filter', 'value'),
    Input('operation-filter', 'value'),
)
def update_dashboard(selected_type, selected_operation):

    filtered_df = df.copy()
    if selected_type != 'All':
        filtered_df = filtered_df[filtered_df['property_type'] == selected_type]
    if selected_operation != 'All':
        filtered_df = filtered_df[filtered_df['operation'] == selected_operation]

    # ---- KPI Cards (use only rows with valid price) ----
    price_valid = filtered_df.dropna(subset=['price_aprox_usd'])

    kpi_style = {
        'padding': '20px', 'border': '1px solid #ddd', 'borderRadius': '10px',
        'textAlign': 'center', 'width': '20%', 'boxShadow': '2px 2px 8px rgba(0,0,0,0.1)',
        'backgroundColor': '#fafafa'
    }

    kpis = [
        html.Div([html.H4("Listings"), html.H2(f"{len(filtered_df):,}")], style=kpi_style),
        html.Div([html.H4("Avg Price (USD)"), html.H2(f"${price_valid['price_aprox_usd'].mean():,.0f}")], style=kpi_style),
        html.Div([html.H4("Avg Price/m²"), html.H2(f"${filtered_df['price_usd_per_m2'].dropna().mean():,.0f}")], style=kpi_style),
        html.Div([html.H4("Avg Surface (m²)"), html.H2(f"{filtered_df['surface_total_in_m2'].dropna().mean():,.0f}")], style=kpi_style),
    ]

    # ---- Map (only rows with valid coordinates AND price) ----
    map_df = filtered_df.dropna(subset=['lat', 'lon', 'price_aprox_usd'])
    map_fig = px.scatter_mapbox(
        map_df, lat='lat', lon='lon',
        color='price_aprox_usd', size_max=10,
        color_continuous_scale='Viridis',
        hover_name='neighborhood',
        hover_data={'price_aprox_usd': ':,.0f', 'lat': False, 'lon': False},
        zoom=10, height=500,
        title='Listings by Location & Price (USD)'
    )
    map_fig.update_layout(mapbox_style='open-street-map', margin={'r': 0, 't': 40, 'l': 0, 'b': 0})

    # ---- Price distribution histogram ----
    hist_fig = px.histogram(
        price_valid, x='price_aprox_usd', nbins=50,
        title='Price Distribution (USD)',
        color_discrete_sequence=['#3498db']
    )
    hist_fig.update_layout(xaxis_title='Price (USD)', yaxis_title='Number of Listings')

    # ---- Average price by property type ----
    type_avg = price_valid.groupby('property_type')['price_aprox_usd'].mean().reset_index()
    type_fig = px.bar(
        type_avg.sort_values('price_aprox_usd'),
        x='price_aprox_usd', y='property_type', orientation='h',
        title='Average Price by Property Type',
        color='price_aprox_usd', color_continuous_scale='Teal'
    )

    # ---- Top 10 neighborhoods by price per m² ----
    neighborhood_avg = filtered_df.dropna(subset=['price_usd_per_m2']).groupby('neighborhood')['price_usd_per_m2'].mean()
    top10_neighborhoods = neighborhood_avg.nlargest(10).reset_index()
    neighborhood_fig = px.bar(
        top10_neighborhoods.sort_values('price_usd_per_m2'),
        x='price_usd_per_m2', y='neighborhood', orientation='h',
        title='Top 10 Most Expensive Neighborhoods (Price per m²)',
        color='price_usd_per_m2', color_continuous_scale='Magenta'
    )

    return kpis, map_fig, hist_fig, type_fig, neighborhood_fig


# ============================================
# 6. RUN
# ============================================
if __name__ == '__main__':
    app.run(debug=True, port=8052)

C:\Users\HP\AppData\Local\Temp\ipykernel_15236\2937522881.py:126: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/

